In [1]:
!pip install -U -q git+https://github.com/huggingface/transformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install -q accelerate qwen-vl-utils bitsandbytes

In [ ]:
# """
# Qwen3-VL 캡션 생성 (LookMatch) — 로컬 / Colab / 구글드라이브 겸용
# ================================================================
# crop된 옷 이미지 + 네이버 상품명(힌트) → 깨끗한 한국어 캡션 생성.
# """

# import os
# import gc
# import json
# from pathlib import Path

# import pandas as pd
# import torch
# from PIL import Image
# from tqdm import tqdm

# def setup_data_dir():
#     env_dir = os.environ.get("LOOKMATCH_DIR")
#     if env_dir:
#         return Path(env_dir)

#     try:
#         import google.colab
#         is_colab = True
#     except ImportError:
#         is_colab = False

#     if is_colab:
#         drive_root = Path("/content/drive/MyDrive")
#         if not drive_root.exists():
#             from google.colab import drive
#             drive.mount("/content/drive")
#         data_dir = drive_root / "clothes_dataset"
#         if not data_dir.exists():
#             data_dir.mkdir(parents=True, exist_ok=True)
#         return data_dir

#     data_dir = Path("clothes_dataset")
#     return data_dir

# DATA_DIR = setup_data_dir()
# IN_CSV = str(DATA_DIR / "metadata_cropped_test.csv")
# OUT_CSV = str(DATA_DIR / "metadata_captioned_test.csv")
# PROGRESS_FILE = str(DATA_DIR / "caption_progress.json")
# CROPS_SUBDIR = "crops"

# BATCH_SIZE = 8
# MAX_NEW_TOKENS = 40
# SAVE_EVERY = 200

# def resolve_image_path(crop_path):
#     """경로 구분자 보정 및 파일명 기반 재탐색"""
#     if pd.isna(crop_path): return None

#     # 1. Windows 백슬래시 및 중복 경로 정리
#     clean_name = str(crop_path).replace('\\', '/').split('/')[-1]

#     # 2. DATA_DIR/crops 폴더 내에서 파일 찾기
#     candidate = DATA_DIR / CROPS_SUBDIR / clean_name
#     if candidate.exists():
#         return str(candidate)

#     # 3. 절대 경로 확인
#     p = Path(crop_path)
#     if p.exists():
#         return str(p)

#     return None

# # PROMPT_TEMPLATE = (
# #     "너는 패션 이미지 검색 데이터셋의 캡션을 만드는 전문가야. "
# #     "아래 옷 이미지를 보고, 이미지-텍스트 검색 모델 학습에 쓸 캡션을 한국어로 만들어줘.\n\n"
# #     "[절대 규칙]\n"
# #     "1. 형식: 서술어(~해, ~다)나 문장 부호(쉼표 등)는 절대 쓰지 마. 오직 띄어쓰기로만 구분된 명사/형용사 조합으로만 출력해.\n"
# #     "2. 어휘: 눈에 보이는 속성(색상, 옷 종류, 핏, 넥라인, 소매, 패턴, 소재)만 쓴다. '편안한', '예쁜' 같은 추상적 표현은 절대 배제한다.\n"
# #     "3. 순서: [색상] [패턴/소재] [핏/실루엣] [디테일] [옷 종류] 순서를 지향한다.\n"
# #     "4. 참고 상품명에 포함된 무의미한 브랜드명이나 오타는 무시하고, 실제 이미지 위주로 작성해.\n\n"
# #     "[예시]\n"
# #     "참고 상품명: 당일발송 [린] 하이웨이스트 린넨 반바지 숏팬츠\n"
# #     "캡션(키워드 나열만 출력): 베이지 린넨 하이웨이스트 핀턱 반바지\n\n"
# #     "참고 상품명: {title}\n"
# #     "캡션(키워드 나열만 출력):"
# # )
# PROMPT_TEMPLATE = (
#     "너는 패션 이미지 검색 데이터셋의 캡션을 만드는 전문가야. "
#     "아래 옷 이미지를 보고, 이미지-텍스트 검색 모델 학습에 쓸 캡션을 한국어로 만들어줘.\n\n"
#     "[규칙]\n"
#     "1. 눈에 보이는 시각적 속성만 쓴다: 색상, 옷 종류, 핏/실루엣, 넥라인, 소매길이, 패턴/디테일, 소재감.\n"
#     "2. '여름', '편안한', '데일리', '예쁜' 같은 추상적·감성적 표현은 절대 쓰지 않는다.\n"
#     "3. 문장이 아니라 명사·형용사 위주의 키워드 나열로 쓴다. (예: '베이지 하이웨이스트 밴딩 와이드 슬랙스')\n"
#     "4. 순서는 [색상] → [패턴/소재] → [핏/실루엣] → [디테일] → [옷 종류] 를 지향한다.\n"
#     "5. 6~12단어 이내로 간결하게. 브랜드명·가격·배송 문구는 넣지 않는다.\n"
#     "6. 상품명은 참고만 하고, 이미지에 실제로 보이는 것을 우선한다.단, 상품명은 반드시 참고해야 한다.\n\n"
#     "참고 상품명: {title}\n\n"
#     "캡션(키워드 나열만 출력):"
# )

# def detect_env():
#     try:
#         import google.colab
#         is_colab = True
#     except ImportError:
#         is_colab = False
#     has_gpu = torch.cuda.is_available()
#     vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if has_gpu else 0
#     return is_colab, has_gpu, vram_gb

# def load_model(vram_gb):
#     from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
#     import torch

#     kwargs = {"device_map": "auto", "dtype": torch.bfloat16}

#     if vram_gb >= 20:
#         model_id = "Qwen/Qwen3-VL-8B-Instruct"
#     elif vram_gb >= 12:
#         model_id = "Qwen/Qwen3-VL-8B-Instruct"
#         # 4bit 양자화 설정 주입
#         kwargs["quantization_config"] = BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_compute_dtype=torch.bfloat16,
#             bnb_4bit_use_double_quant=True,
#             bnb_4bit_quant_type="nf4"
#         )
#     else:
#         model_id = "Qwen/Qwen3-VL-2B-Instruct"

#     model = AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
#     processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=768*28*28)
#     processor.tokenizer.padding_side = "left"

#     return model.eval(), processor

# def clean_caption(text):
#     text = text.strip()
#     for junk in ['"', "'", "`", "*", ":", "\\n"]: text = text.replace(junk, " ")
#     return " ".join(text.split())

# def caption_batch(model, processor, rows):
#     messages_list = []
#     valid_idx = []
#     for i, row in enumerate(rows):
#         img_path = resolve_image_path(row["crop_path"])
#         if img_path is None: continue
#         messages_list.append([{
#             "role": "user",
#             "content": [{"type": "image", "image": img_path}, {"type": "text", "text": PROMPT_TEMPLATE.format(title=str(row.get("caption", ""))[:100])}],
#         }])
#         valid_idx.append(i)
#     if not messages_list: return {}

#     from qwen_vl_utils import process_vision_info
#     texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_list]
#     image_inputs, video_inputs = process_vision_info(messages_list)
#     inputs = processor(text=texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         gen_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
#     trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, gen_ids)]
#     outputs = processor.batch_decode(trimmed, skip_special_tokens=True)
#     return {valid_idx[i]: clean_caption(cap) for i, cap in enumerate(outputs)}

# def main():
#     is_colab, has_gpu, vram_gb = detect_env()
#     df = pd.read_csv(IN_CSV)
#     captions = {}
#     if Path(PROGRESS_FILE).exists():
#         with open(PROGRESS_FILE) as f: captions = {int(k): v for k, v in json.load(f).items()}

#     model, processor = load_model(vram_gb)
#     todo = [i for i in range(len(df)) if i not in captions]

#     for start in tqdm(range(0, len(todo), BATCH_SIZE)):
#         batch_idx = todo[start:start + BATCH_SIZE]
#         try:
#             res = caption_batch(model, processor, [df.iloc[i].to_dict() for i in batch_idx])
#             for idx_in_batch, cap in res.items(): captions[batch_idx[idx_in_batch]] = cap
#         except Exception as e: print(f"Error: {e}"); continue

#         if (start // BATCH_SIZE) % 10 == 0:
#             with open(PROGRESS_FILE, "w") as f: json.dump({str(k): v for k, v in captions.items()}, f)

#     df["qwen_caption"] = df.index.map(lambda i: captions.get(i, ""))
#     df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
#     print(f"Done! Saved to {OUT_CSV}")

# if __name__ == "__main__":
#     main()

In [4]:
"""
Qwen3-VL 캡션 생성 (LookMatch) — 로컬 / Colab / 구글드라이브 겸용
================================================================
crop된 옷 이미지 + 네이버 상품명(힌트) → 깨끗한 한국어 캡션 생성.
네이버 제목의 SEO 키워드 나열을 CLIP 학습에 좋은 캡션으로 정제.

특징:
  - 실행 환경 자동 감지 (로컬 / Colab / 구글드라이브)
    · 로컬: 현재 폴더의 clothes_dataset/ 사용
    · Colab: 구글드라이브 자동 연결 → MyDrive/lookmatch/ 사용
    · 환경변수 LOOKMATCH_DIR 로 경로 직접 지정도 가능
  - VRAM에 따라 Qwen3-VL 모델 자동 선택 (8B / 4B / 2B)
  - 배치 처리로 속도 ↑
  - 이어받기 (중간에 끊겨도 재실행하면 이어서)
  - 이미지 경로 자동 보정

모델 (VRAM 기준 자동):
  20GB+ → Qwen3-VL-8B-Instruct  (bf16, 품질 우선)
  12GB+ → Qwen3-VL-8B + 4bit    (8B 4bit ≈ 6GB)
  8GB+  → Qwen3-VL-4B-Instruct  (균형)
  그 미만 → Qwen3-VL-2B-Instruct (최경량)

입력:  <데이터폴더>/metadata_cropped.csv (crop_path, caption)
출력:  <데이터폴더>/metadata_captioned.csv (+ qwen_caption 컬럼)

사용법:
  [로컬]
    pip install -U git+https://github.com/huggingface/transformers
    pip install accelerate qwen-vl-utils bitsandbytes pillow pandas tqdm torch
    python generate_captions.py

  [Colab]
    !pip install -U -q git+https://github.com/huggingface/transformers
    !pip install -q accelerate qwen-vl-utils bitsandbytes
    !python generate_captions.py
    # → 드라이브 자동 연결, MyDrive/lookmatch/ 사용

  [경로 직접 지정]
    LOOKMATCH_DIR=/my/path python generate_captions.py

※ Qwen3-VL은 최신 transformers 필요 (구버전이면 모델 로드 실패).
"""

import os
import gc
import json
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
import re

# ============================================================
# 경로 설정 — 로컬 / Colab / 구글드라이브 자동 감지
# ============================================================
# 우선순위:
#   1) 환경변수 LOOKMATCH_DIR 이 있으면 그걸 사용
#   2) Colab이고 드라이브에 lookmatch 폴더가 있으면 드라이브 사용
#   3) Colab이면 드라이브 연결 후 드라이브 사용
#   4) 로컬이면 현재 폴더의 clothes_dataset 사용
def setup_data_dir():
    # 1) 환경변수 우선 (사용자가 직접 지정 가능)
    env_dir = os.environ.get("LOOKMATCH_DIR")
    if env_dir:
        print(f"데이터 경로(환경변수): {env_dir}")
        return Path(env_dir)

    # Colab 여부 확인
    try:
        import google.colab  # noqa
        is_colab = True
    except ImportError:
        is_colab = False

    if is_colab:
        # 2~3) Colab → 드라이브 연결
        drive_root = Path("/content/drive/MyDrive")
        if not drive_root.exists():
            print("구글 드라이브 연결 중... (인증 창에서 허용해주세요)")
            from google.colab import drive
            drive.mount("/content/drive")
        # 드라이브 안 lookmatch 폴더 (없으면 생성)
        data_dir = drive_root / "clothes_dataset"
        data_dir.mkdir(parents=True, exist_ok=True)
        print(f"데이터 경로(구글드라이브): {data_dir}")
        return data_dir

    # 4) 로컬
    data_dir = Path("clothes_dataset")
    print(f"데이터 경로(로컬): {data_dir.resolve()}")
    return data_dir


DATA_DIR = setup_data_dir()
IN_CSV = str(DATA_DIR / "metadata_cropped_test.csv")
OUT_CSV = str(DATA_DIR / "metadata_captioned_test.csv")
# IN_CSV = str(DATA_DIR / "metadata_cropped.csv")
# OUT_CSV = str(DATA_DIR / "metadata_captioned.csv")
PROGRESS_FILE = str(DATA_DIR / "caption_progress.json")

# crop 이미지 경로가 다른 환경에서 만들어졌을 때 보정용
# (예: 로컬에서 크롭 → 드라이브에서 캡션 → crop_path가 로컬 경로로 남아있음)
# CSV의 crop_path가 절대경로로 안 맞으면, 파일명만 떼어 DATA_DIR/crops 에서 찾음
CROPS_SUBDIR = "crops"

BATCH_SIZE = 8           # 배치 크기 (VRAM 부족하면 줄이기)
MAX_NEW_TOKENS = 50      # 출력 길이 제한 (짧게 = 빠름, 간결)
SAVE_EVERY = 200         # N개마다 중간 저장 (이어받기용)
LIMIT = 0                # 테스트용: 0이면 전체, 100이면 앞 100개만 처리


def resolve_image_path(crop_path):
    """crop_path가 현재 환경에서 안 맞을 때 파일명으로 재탐색.
    윈도우(\\)·리눅스(/) 경로 혼용도 처리."""
    crop_path = str(crop_path)
    # 1) 있는 그대로 존재하면 사용
    if Path(crop_path).exists():
        return str(Path(crop_path))
    # 2) 윈도우 백슬래시(\)와 리눅스 슬래시(/) 모두에서 파일명만 추출
    #    (예: "clothes_dataset\crops\abc.jpg" → "abc.jpg")
    fname = crop_path.replace("\\", "/").split("/")[-1]
    # 3) DATA_DIR/crops 에서 파일명으로 찾기
    candidate = DATA_DIR / CROPS_SUBDIR / fname
    if candidate.exists():
        return str(candidate)
    return None  # 못 찾음

# 프롬프트 — CLIP 파인튜닝 최적화 (간결 버전)
# 규칙을 최소화하고 예시로 형식을 보여줌 (복잡하면 작은 모델이 프롬프트를 되풀이함)
# PROMPT_TEMPLATE = (
#     "너는 패션 이미지 검색 데이터셋의 캡션을 만드는 전문가야. "
#     "아래 옷 이미지를 보고, 이미지-텍스트 검색 모델 학습에 쓸 캡션을 한국어로 만들어줘.\n\n"
#     "[규칙]\n"
#     "1. 눈에 보이는 시각적 속성만 쓴다: 색상, 옷 종류, 핏/실루엣, 넥라인, 소매길이, 패턴/디테일, 소재감.\n"
#     "2. '여름', '편안한', '데일리', '예쁜' 같은 추상적·감성적 표현은 절대 쓰지 않는다.\n"
#     "3. 문장이 아니라 명사·형용사 위주의 키워드 나열로 쓴다. (예: '베이지 하이웨이스트 밴딩 와이드 슬랙스')\n"
#     "4. 순서는 [색상] → [패턴/소재] → [핏/실루엣] → [디테일] → [옷 종류] 를 지향한다.\n"
#     "5. 6~12단어 이내로 간결하게. 브랜드명·가격·배송 문구는 넣지 않는다.\n"
#     "6. 상품명을 참고해서 작성하되, 이미지에 실제로 보이는 것을 우선으로 작성한다.\n\n"
#     "참고 상품명: {title}\n\n"
#     "캡션(키워드 나열만 출력):"
# )
# 프롬프트 — 멀티모달 Chain-of-Thought (CoT)
# 바로 캡션을 뱉지 않고, 이미지를 속성별로 단계 분석 후 최종 캡션 생성
# → 색상/종류/핏을 하나씩 짚어 실수(종류 누락·예시 복사·환각) 감소
# 출력 형식: "분석: ... / 캡션: ..." → 후처리에서 '캡션:' 뒤만 추출
PROMPT_TEMPLATE = (
    "너는 패션 이미지 검색 데이터셋 캡션 전문가야. "
    "아래 옷 이미지를 단계적으로 분석한 뒤, 검색 모델 학습용 한국어 캡션을 만들어줘.\n\n"
    "다음 순서로 이미지를 관찰하고 각 항목을 한국어로 짧게 채워줘. "
    "이미지에 실제로 보이는 것만 쓰고, 안 보이거나 해당 없으면 '없음'이라 써.\n"
    "1. 색상: (주요 색상)\n"
    "2. 옷 종류: (예: 슬랙스, 니트, 코트 — 상품명도 참고)\n"
    "3. 핏/실루엣: (예: 와이드, 오버핏, 슬림)\n"
    "4. 패턴/소재: (예: 무지, 체크, 데님, 니트)\n"
    "5. 디테일: (예: 밴딩, 하이웨이스트, 라운드넥 — 보이는 것만)\n\n"
    "그런 다음, 위 항목 중 '없음'이 아닌 것만 모아 "
    "[색상] [패턴/소재] [핏] [디테일] [옷 종류] 순서로 "
    "띄어쓰기로 이어 붙여 최종 캡션을 만들어. "
    "쉼표·문장·추상어(캐주얼/데일리)·브랜드명은 쓰지 마. "
    "반드시 마지막은 옷 종류로 끝내.\n\n"
    "참고 상품명: {title}\n\n"
    "아래 형식으로만 답해:\n"
    "분석: 1.색상=... 2.종류=... 3.핏=... 4.패턴=... 5.디테일=...\n"
    "캡션: (최종 키워드 [색상] [패턴/소재] [핏] [디테일] [옷 종류] 순서로 나열)"
)

# ============================================================
# 환경 감지
# ============================================================
def detect_env():
    """Colab인지, GPU 있는지, VRAM 얼마인지 감지"""
    try:
        import google.colab  # noqa
        is_colab = True
    except ImportError:
        is_colab = False

    has_gpu = torch.cuda.is_available()
    vram_gb = 0
    if has_gpu:
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

    print(f"환경: {'Colab' if is_colab else '로컬'}")
    print(f"GPU: {'있음' if has_gpu else '없음 (CPU - 매우 느림)'}")
    if has_gpu:
        print(f"VRAM: {vram_gb:.1f}GB ({torch.cuda.get_device_name(0)})")
    return is_colab, has_gpu, vram_gb


# ============================================================
# 모델 선택 & 로드 (VRAM에 맞게)
# ============================================================
def load_model(vram_gb):
    """VRAM에 따라 Qwen3-VL 모델 크기·양자화 자동 선택"""
    from transformers import AutoModelForImageTextToText, AutoProcessor

    # VRAM 기준 Qwen3-VL 모델 선택
    # (bf16 기준 대략: 8B~18GB, 4B~9GB, 2B~5GB / 8B 4bit~6GB)
    if vram_gb >= 20:
        model_id = "Qwen/Qwen3-VL-8B-Instruct"
        quant = "4bit"
        print(f"→ Qwen3-VL-8B (VRAM 충분, 품질 우선)")
    elif vram_gb >= 12:
        model_id = "Qwen/Qwen3-VL-8B-Instruct"
        quant = "4bit"
        print(f"→ Qwen3-VL-8B + 4bit 양자화 (8B 4bit ≈ 6GB)")
    elif vram_gb >= 8:
        model_id = "Qwen/Qwen3-VL-4B-Instruct"
        quant = None
        print(f"→ Qwen3-VL-4B (균형)")
    else:
        model_id = "Qwen/Qwen3-VL-2B-Instruct"
        quant = None
        print(f"→ Qwen3-VL-2B (VRAM 제한, 최경량)")

    kwargs = {"device_map": "auto", "torch_dtype": torch.bfloat16}

    # 4bit 양자화 (VRAM 절약)
    if quant == "4bit":
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    # 이미지 토큰 수 제한 (속도·메모리)
    min_pixels = 256 * 28 * 28
    max_pixels = 768 * 28 * 28

    model = AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
    processor = AutoProcessor.from_pretrained(
        model_id, min_pixels=min_pixels, max_pixels=max_pixels
    )
    # 배치 생성(decoder)에서는 left padding이 필수
    # (right padding이면 PAD 토큰 위치에서 생성이 시작돼 출력이 깨짐)
    if hasattr(processor, "tokenizer"):
        processor.tokenizer.padding_side = "left"
    model.eval()
    return model, processor


# ============================================================
# 캡션 정제 (후처리)
# ============================================================
def clean_caption(text):
    """Qwen 출력에서 불필요한 부분 제거 → 깨끗한 키워드 나열만"""
    text = text.strip()
    # 접두어 제거 (모델이 붙이는 군더더기)
    prefixes = ["캡션:", "캡션", "답:", "특징:", "키워드:",
                "이 옷은", "이 옷", "이미지에는", "이미지에", "사진에는", "사진에",
                "옷 종류:", "색상:"]
    for p in prefixes:
        if text.startswith(p):
            text = text[len(p):].strip()
    # 따옴표·기호 제거
    for junk in ['"', "'", "`", "```", "*", "-", "•", "·", ":", "\n"]:
        text = text.replace(junk, " ")
    text = text.replace(",", " ").replace("，", " ")
    text = re.sub(r"\S*\s*없음", "", text)
    text = re.sub(r"\S*\s*없는", "", text)
    # 문장 끝 표현 제거 (혹시 문장으로 나온 경우)
    for tail in ["입니다", "이에요", "예요", "습니다", "같습니다", "보입니다","있음", "이며", "하며"]:
        text = text.replace(tail, "")
    # 공백 정리
    text = " ".join(text.split())
    return text.strip(" .·:")[:100]


# ============================================================
# 배치 캡션 생성
# ============================================================
def caption_batch(model, processor, rows):
    """이미지+제목 배치 → 캡션 리스트"""
    messages_list = []
    valid_idx = []
    for i, row in enumerate(rows):
        img_path = resolve_image_path(row["crop_path"])
        if img_path is None:
            continue
        title = str(row.get("caption", ""))[:100]  # 네이버 제목(힌트)
        messages_list.append([{
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": PROMPT_TEMPLATE.format(title=title)},
            ],
        }])
        valid_idx.append(i)

    if not messages_list:
        return {}

    # 배치 전처리
    from qwen_vl_utils import process_vision_info
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
             for m in messages_list]
    image_inputs, video_inputs = process_vision_info(messages_list)
    inputs = processor(
        text=texts, images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    # 생성
    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, gen_ids)]
    outputs = processor.batch_decode(trimmed, skip_special_tokens=True,
                                     clean_up_tokenization_spaces=False)

    # idx → 캡션 매핑
    result = {}
    for local_i, cap in zip(valid_idx, outputs):
        result[local_i] = clean_caption(cap)
    return result


# ============================================================
# 메인
# ============================================================
def main():
    is_colab, has_gpu, vram_gb = detect_env()
    if not has_gpu:
        print("\n⚠️ GPU가 없어 매우 느립니다. Colab GPU 런타임을 권장합니다.")

    # 테스트 모드면 출력/진행 파일을 별도로 (전체 실행과 안 섞이게)
    out_csv = OUT_CSV
    progress_file = PROGRESS_FILE
    if LIMIT > 0:
        out_csv = OUT_CSV.replace(".csv", "_test.csv")
        progress_file = PROGRESS_FILE.replace(".json", "_test.json")

    df = pd.read_csv(IN_CSV)
    if LIMIT > 0:
        df = df.head(LIMIT).copy()
        print(f"[테스트 모드] 앞 {LIMIT}개만 처리 → {out_csv}")
    print(f"\n총 {len(df)}개 캡션 생성 대상\n")

    # 이어받기: 이미 처리한 인덱스 로드
    done = set()
    captions = {}
    if Path(progress_file).exists():
        with open(progress_file, encoding="utf-8") as f:
            saved = json.load(f)
            captions = {int(k): v for k, v in saved.items()}
            done = set(captions.keys())
        print(f"이어받기: {len(done)}개 이미 완료\n")

    # 모델 로드
    model, processor = load_model(vram_gb)

    # 배치 처리
    todo = [i for i in range(len(df)) if i not in done]
    for start in tqdm(range(0, len(todo), BATCH_SIZE), desc="캡션 생성"):
        batch_idx = todo[start:start + BATCH_SIZE]
        rows = [df.iloc[i].to_dict() for i in batch_idx]

        try:
            batch_result = caption_batch(model, processor, rows)
            for local_i, cap in batch_result.items():
                captions[batch_idx[local_i]] = cap
        except torch.cuda.OutOfMemoryError:
            print(f"\nOOM! BATCH_SIZE를 줄이세요 (현재 {BATCH_SIZE}). 진행분 저장.")
            torch.cuda.empty_cache()
            break
        except Exception as e:
            print(f"\n배치 오류(건너뜀): {e}")
            continue

        # 중간 저장 (이어받기용)
        if (start // BATCH_SIZE) % (SAVE_EVERY // BATCH_SIZE + 1) == 0:
            with open(progress_file, "w", encoding="utf-8") as f:
                json.dump({str(k): v for k, v in captions.items()}, f, ensure_ascii=False)
            gc.collect()
            if has_gpu:
                torch.cuda.empty_cache()

    # 최종 저장
    with open(progress_file, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in captions.items()}, f, ensure_ascii=False)

    df["qwen_caption"] = df.index.map(lambda i: captions.get(i, ""))
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")

    # 요약 + 샘플
    n_done = sum(1 for v in captions.values() if v)
    print(f"\n{'='*55}")
    print(f"캡션 생성 완료: {n_done}/{len(df)}개")
    print(f"저장: {out_csv}")
    print(f"\n[샘플 5개 — 네이버 제목 → Qwen 캡션]")
    shown = 0
    for i in range(len(df)):
        if captions.get(i):
            orig = str(df.iloc[i].get("caption", ""))[:40]
            print(f"  원본: {orig}")
            print(f"  정제: {captions[i]}\n")
            shown += 1
            if shown >= 5:
                break
    print("="*55)


if __name__ == "__main__":
    main()

데이터 경로(구글드라이브): /content/drive/MyDrive/clothes_dataset
환경: Colab
GPU: 있음
VRAM: 23.7GB (NVIDIA L4)

총 100개 캡션 생성 대상

이어받기: 100개 이미 완료

→ Qwen3-VL-8B (VRAM 충분, 품질 우선)


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

캡션 생성: 0it [00:00, ?it/s]


캡션 생성 완료: 100/100개
저장: /content/drive/MyDrive/clothes_dataset/metadata_captioned_test.csv

[샘플 5개 — 네이버 제목 → Qwen 캡션]
  원본: 여름 간절기 린넨 배기 바지 밴딩 스판 무지 여성 쿨 팬츠
  정제: 분석 1.색상=노란색 올리브 베이지 흰색 2.종류=바지 3.핏=배기 4.패턴=무지

  원본: 개미다리로 보이는 여자 린넨 반 바지 바지 밴딩 여름 바지 빅사이즈 슬렉
  정제: 분석 1.색상=베이지 2.종류=반바지 3.핏=와이드 4.패턴=무지 5.디테일=밴딩 캡션

  원본: 냉장고 일자 세미 와이드 밴딩팬츠
  정제: 분석 1.색상=검정 2.종류=팬츠 3.핏=와이드 4.패턴=무지 5.디테일=밴딩 캡션 검정

  원본: 여름 여자 시원한 쿨 스판 세미와이드 카고 조거 팬츠 냉장고 편안한 냉감
  정제: 분석 1.색상=검정 그레이 올리브 2.종류=팬츠 3.핏=세미와이드 4.패턴=무지 5.디테

  원본: 폴햄 쿨링 린렌라이크 밴딩팬츠 PHG2PP2089
  정제: 분석 1.색상=회색 2.종류=팬츠 3.핏=와이드 4.패턴=무지 5.디테일=밴딩 캡션 회색

